# 다이캐스팅 불량 판별 모델 (Product Type 2)

## 프로젝트 개요
- **데이터**: Product Type 2
- **모델**: XGBoost
- **해석**: SHAP 분석

## 특징
- 간단한 이진 분류
- SHAP으로 모델 해석
- 결측치 자동 처리

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

print("라이브러리 import 완료")

## 1. 데이터 로드 (MultiIndex Header)

In [ ]:
# MultiIndex로 CSV 로드
df = pd.read_csv("product_type_2.csv", header=[0, 1])

print(f"데이터 크기: {df.shape[0]:,}행 × {df.shape[1]}열")
print(f"결측치: {df.isnull().sum().sum()}개")
print(f"\n컬럼 구조 샘플:")
print(df.columns[:5].tolist())

df.head(3)

## 2. 타겟 변수 추출 (is_defect)

In [ ]:
# 타겟 추출
defect_cols = [col for col in df.columns if col[0] == 'defects']
y = (df[defect_cols] > 0).any(axis=1).astype(int)

print("타겟 분포:")
print(y.value_counts())
print(f"\n불량률: {y.mean()*100:.2f}%")

# 시각화
fig, ax = plt.subplots(figsize=(6, 4))
counts = y.value_counts().sort_index()
ax.bar([0, 1], counts, color=['green', 'red'], alpha=0.7)
ax.set_xlabel('클래스')
ax.set_ylabel('개수')
ax.set_title('불량 유무 분포', fontsize=14, fontweight='bold')
ax.set_xticks([0, 1])
ax.set_xticklabels(['정상 (0)', '불량 (1)'])
ax.grid(axis='y', alpha=0.3)
for i, v in enumerate(counts):
    ax.text(i, v + 20, f"{v}\n({v/len(y)*100:.1f}%)", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Feature 추출 (Process + Sensor)

Data Leakage 방지: defects 컬럼 제외

In [ ]:
# Process Features (shot 제외)
process_cols = [col for col in df.columns if col[0] == 'process' and col[1] != 'shot']

# Sensor Features
sensor_cols = [col for col in df.columns if col[0] == 'sensor']

print(f"✓ Process 변수: {len(process_cols)}개")
print(f"✓ Sensor 변수: {len(sensor_cols)}개")

# Feature names (보기 쉽게)
process_names = [col[1] for col in process_cols]
sensor_names = [col[1] for col in sensor_cols]

print(f"\nProcess: {process_names[:5]}...")
print(f"Sensor: {sensor_names[:5]}...")

# X 생성
X = pd.concat([df[process_cols], df[sensor_cols]], axis=1)

# 컬럼명 단순화 (MultiIndex → single level)
X.columns = process_names + sensor_names

print(f"\nX shape: {X.shape}")
print(f"y shape: {y.shape}")

## 4. Train/Test 분리 (80/20)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"\nTrain 불량률: {y_train.mean()*100:.2f}%")
print(f"Test 불량률: {y_test.mean()*100:.2f}%")

## 5. 결측치 처리

In [ ]:
# 결측치 확인
print(f"Train 결측치: {X_train.isnull().sum().sum()}개")
print(f"\n결측치가 있는 컬럼:")
null_cols = X_train.isnull().sum()
null_cols = null_cols[null_cols > 0].sort_values(ascending=False)
print(null_cols)

# 중앙값 대체
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

print(f"\n결측치 처리 완료")
print(f"X_train_imp shape: {X_train_imp.shape}")

## 6. XGBoost 모델 학습

In [ ]:
# 클래스 불균형 계산
neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
scale_pos_weight = neg / pos

print(f"정상: {neg:,}개")
print(f"불량: {pos:,}개")
print(f"비율: {scale_pos_weight:.2f}:1")
print(f"scale_pos_weight: {scale_pos_weight:.3f}")

# XGBoost 모델
model = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1,
    eval_metric='logloss'
)

print(f"\n모델 학습 중")
model.fit(X_train_imp, y_train)
print("학습 완료")

## 7. 모델 평가

In [ ]:
y_pred = model.predict(X_test_imp)
y_proba = model.predict_proba(X_test_imp)[:, 1]

print("="*60)
print("분류 성능 리포트")
print("="*60)
print(classification_report(y_test, y_pred, target_names=['정상', '불량']))

roc_auc = roc_auc_score(y_test, y_proba)
print(f"\nROC-AUC: {roc_auc:.4f}")

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 혼동 행렬
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['정상', '불량'], yticklabels=['정상', '불량'])
axes[0].set_title('혼동 행렬', fontsize=14, fontweight='bold')
axes[0].set_ylabel('실제')
axes[0].set_xlabel('예측')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, linewidth=2, label=f'ROC (AUC={roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Threshold 튜닝

현재는 확률이 0.5 이상이면 불량이라고 판단중

불량이 적은 데이터에서는 0.5가 너무 높아서 불량을 많이 놓칠 수 있음. 

F1-score 중점 기준 최적 threshold를 자동으로 찾는 최종 튜닝 모델

In [ ]:
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve
)
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# 예측 확률
y_proba = model.predict_proba(X_test_imp)[:, 1]

# --------------------------------------------------
# 1. F1-score가 최대가 되는 threshold 찾기
# --------------------------------------------------
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)

# precision_recall_curve의 precisions, recalls는 thresholds보다 길이가 1 큼
f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)

best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
best_f1 = f1_scores[best_idx]
best_precision = precisions[best_idx]
best_recall = recalls[best_idx]

print("="*60)
print("F1-score 기준 최적 threshold 탐색 결과")
print("="*60)
print(f"최적 threshold : {best_threshold:.4f}")
print(f"Precision      : {best_precision:.4f}")
print(f"Recall         : {best_recall:.4f}")
print(f"F1-score       : {best_f1:.4f}")

# --------------------------------------------------
# 2. 최적 threshold로 최종 예측
# --------------------------------------------------
y_pred = (y_proba >= best_threshold).astype(int)

print("\n" + "="*60)
print("분류 성능 리포트")
print("="*60)
print(classification_report(y_test, y_pred, target_names=['정상', '불량']))

roc_auc = roc_auc_score(y_test, y_proba)
print(f"\nROC-AUC: {roc_auc:.4f}")

# --------------------------------------------------
# 3. 시각화
# --------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 혼동 행렬
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    ax=axes[0],
    xticklabels=['정상', '불량'],
    yticklabels=['정상', '불량']
)
axes[0].set_title(f'혼동 행렬 (threshold={best_threshold:.3f})', fontsize=14, fontweight='bold')
axes[0].set_ylabel('실제')
axes[0].set_xlabel('예측')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, linewidth=2, label=f'ROC (AUC={roc_auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# threshold별 F1-score 시각화
plt.figure(figsize=(8, 5))
plt.plot(thresholds, f1_scores, linewidth=2)
plt.axvline(best_threshold, linestyle='--', label=f'Best threshold = {best_threshold:.3f}')
plt.xlabel('Threshold')
plt.ylabel('F1-score')
plt.title('Threshold별 F1-score 변화', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 8. Feature Importance

In [ ]:
importance = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

print("="*60)
print("Feature Importance TOP 15")
print("="*60)
print(importance.head(15))

plt.figure(figsize=(10, 8))
importance.head(20).sort_values().plot(kind='barh', color='steelblue', alpha=0.8)
plt.xlabel('Importance Score')
plt.title('Feature Importance (TOP 20)', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 9. SHAP 분석

In [ ]:
import shap

print("SHAP Explainer 생성 중")

X_test_df = pd.DataFrame(X_test_imp, columns=X.columns, index=X_test.index)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_df)

if isinstance(shap_values, list):
    shap_values = shap_values[1] if len(shap_values) == 2 else shap_values[0]
shap_values = np.asarray(shap_values)
if shap_values.ndim == 3:
    shap_values = shap_values[:, :, -1]

print(f"\nSHAP values shape: {shap_values.shape}")
print("SHAP 계산 완료")

### 9.1 SHAP Summary (Bar)

불량 판별에 중요한 변수

In [ ]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test_df, plot_type="bar", max_display=20, show=False)
plt.title("불량 판별 주요 변수 (SHAP)", fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

shap_imp = pd.Series(np.abs(shap_values).mean(axis=0), index=X.columns).sort_values(ascending=False)
print("\nSHAP 중요도 TOP 10:")
print(shap_imp.head(10))